# MINE: Regime III, ProtMamba (Geometric Vacuity)

Separate notebook for ProtMamba evaluation. Saves results to `./results/mine/`
for combined analysis.

In [ ]:
import os
import numpy as np

PHASE = 'full'  # 'quick' or 'full'

OUTPUT_BASE = './results/mine_v2_three_regimes/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIG = {
    'quick': {
        'n_sequences': 100,
        'seq_lengths': [100, 200],
        'mine_epochs': 200,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 3,
    },
    'full': {
        'n_sequences': 200,
        'seq_lengths': [100, 200, 400],
        'mine_epochs': 500,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 5,
    },
}

RANDOM_SEED = 320
config = CONFIG[PHASE]

print(f"Phase: {PHASE.upper()}")
print(f"Sequences per class: {config['n_sequences']}")
print(f"Sequence lengths: {config['seq_lengths']}")
print(f"MINE epochs: {config['mine_epochs']}")
print(f"MINE independent runs: {config['n_runs']}")
print(f"Results: {RESULTS_DIR}")

## Install Dependencies

In [ ]:
import os, sys

print("Installing base dependencies...")
!pip install -q matplotlib seaborn pandas scipy scikit-learn numpy

!pip install -q biopython

print("Installing ProtMamba dependencies...")
!pip install -q causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -3

# Clone ProtMamba
if not os.path.exists('ProtMamba-ssm'):
    !git clone https://github.com/Bitbol-Lab/ProtMamba-ssm.git
!pip install -q -e ProtMamba-ssm/

# Download ProtMamba weights
PROTMAMBA_WEIGHTS_DIR = 'protmamba_weights'
if not os.path.exists(PROTMAMBA_WEIGHTS_DIR):
    os.makedirs(PROTMAMBA_WEIGHTS_DIR, exist_ok=True)
    !wget -q -O protmamba_weights.zip https://github.com/Bitbol-Lab/ProtMamba-ssm/releases/download/v1.0/ProtMamba_model-weights.zip
    !unzip -q -o protmamba_weights.zip -d {PROTMAMBA_WEIGHTS_DIR}
    !rm protmamba_weights.zip

import torch, numpy as np
print(f"\ntorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

for name, mod in [('mamba_ssm', 'mamba_ssm'), ('causal_conv1d', 'causal_conv1d')]:
    try:
        __import__(mod)
        print(f"  {name}: OK")
    except ImportError as e:
        print(f"  {name}: FAILED ({e})")

print("Done!")

## MINE Core Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MINEStatisticsNetwork(nn.Module):
    # Statistics network T_theta for MINE.
    # Takes concatenated (x, x_hat) as input and outputs a scalar.
    def __init__(self, input_dim, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def mine_estimate(T_net, joint_batch, marginal_batch):
    # Compute the Donsker-Varadhan lower bound on MI.
    # I(X;X_hat) >= E_joint[T(x,x_hat)] - log(E_marginal[exp(T(x,x_hat))])
    t_joint = T_net(joint_batch)
    t_marginal = T_net(marginal_batch)
    first_term = t_joint.mean()
    second_term = torch.logsumexp(t_marginal, dim=0) - np.log(t_marginal.shape[0])
    mi_lb = first_term - second_term
    return mi_lb


def run_mine(X_bio, X_emb, n_epochs, batch_size, lr, hidden_dims, seed,
             device='cuda', verbose=True):
    # Run a single MINE estimation.
    torch.manual_seed(seed)
    np.random.seed(seed)

    N = X_bio.shape[0]
    d_bio = X_bio.shape[1]
    d_emb = X_emb.shape[1]
    input_dim = d_bio + d_emb

    X_bio_t = torch.FloatTensor(
        (X_bio - X_bio.mean(0)) / (X_bio.std(0) + 1e-8)
    ).to(device)
    X_emb_t = torch.FloatTensor(
        (X_emb - X_emb.mean(0)) / (X_emb.std(0) + 1e-8)
    ).to(device)

    T_net = MINEStatisticsNetwork(input_dim, hidden_dims).to(device)
    optimizer = optim.Adam(T_net.parameters(), lr=lr)

    mi_estimates = []

    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        marginal_perm = torch.randperm(N)
        epoch_mi = []

        for i in range(0, N - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            marginal_idx = marginal_perm[i:i + batch_size]

            x_bio_batch = X_bio_t[idx]
            x_emb_batch = X_emb_t[idx]

            joint = torch.cat([x_bio_batch, x_emb_batch], dim=1)
            x_emb_marginal = X_emb_t[marginal_idx]
            marginal = torch.cat([x_bio_batch, x_emb_marginal], dim=1)

            mi_lb = mine_estimate(T_net, joint, marginal)
            loss = -mi_lb
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(T_net.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_mi.append(mi_lb.item())

        mean_mi = np.mean(epoch_mi)
        mi_estimates.append(mean_mi)

        if verbose and (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs}: MI_lb = {mean_mi:.4f}")

    tail = max(1, len(mi_estimates) // 10)
    final_mi = np.mean(mi_estimates[-tail:])
    return mi_estimates, final_mi


def run_mine_with_ci(X_bio, X_emb, config, device='cuda', label=''):
    # Run MINE multiple times and compute confidence intervals.
    all_traces = []
    all_finals = []

    for run_idx in range(config['n_runs']):
        seed = RANDOM_SEED + run_idx * 100
        if label:
            print(f"\n  [{label}] Run {run_idx+1}/{config['n_runs']} (seed={seed})")
        traces, final_mi = run_mine(
            X_bio, X_emb,
            n_epochs=config['mine_epochs'],
            batch_size=config['mine_batch_size'],
            lr=config['mine_lr'],
            hidden_dims=config['mine_hidden'],
            seed=seed,
            device=device,
            verbose=(run_idx == 0),
        )
        all_traces.append(traces)
        all_finals.append(final_mi)
        print(f"    Final MI = {final_mi:.4f}")

    mean_mi = np.mean(all_finals)
    std_mi = np.std(all_finals)
    print(f"\n  [{label}] MI = {mean_mi:.4f} +/- {std_mi:.4f}")
    return all_traces, all_finals, mean_mi, std_mi


print("MINE implementation ready")
print(f"  Architecture: input -> {config['mine_hidden']} -> 1")
print(f"  Training: {config['mine_epochs']} epochs, lr={config['mine_lr']}")

## MINE Sanity Check

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MINE SANITY CHECK: Correlated Gaussians with known MI')
print('='*70)

sanity_results = []
for rho in [0.0, 0.3, 0.6, 0.9]:
    N_sanity = 2000
    rng_sc = np.random.default_rng(RANDOM_SEED)
    z1 = rng_sc.standard_normal(N_sanity)
    z2 = rng_sc.standard_normal(N_sanity)
    x_sanity = z1.reshape(-1, 1).astype(np.float32)
    y_sanity = (rho * z1 + np.sqrt(1 - rho**2) * z2).reshape(-1, 1).astype(np.float32)

    mi_true = -0.5 * np.log(1 - rho**2) if rho != 0 else 0.0

    traces_sc, mi_est = run_mine(
        x_sanity, y_sanity, n_epochs=300, batch_size=64,
        lr=1e-4, hidden_dims=[128, 64], seed=RANDOM_SEED,
        device=device, verbose=False,
    )
    sanity_results.append({'rho': rho, 'mi_true': mi_true, 'mi_est': mi_est})
    tol_ok = abs(mi_est - mi_true) < max(0.15, 0.3 * mi_true)
    status = 'PASS' if tol_ok else 'FAIL'
    print(f'  rho={rho:.1f}: MI_true={mi_true:.4f}, MI_est={mi_est:.4f}  [{status}]')

print('\nIf estimates track the true values, MINE is calibrated.')
print('Proceed to real model evaluation.\n')

## Biological Ground Truth (Proteins)

In [ ]:
import urllib.request

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

ECOLI_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:83333)+AND+(reviewed:true)&size=500'
HUMAN_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:9606)+AND+(reviewed:true)&size=500'


def download_proteins(url, n_sequences, min_length, seed, cache_tag):
    '''Download and cache protein sequences from UniProt.'''
    cache_file = f'{CACHE_DIR}/{cache_tag}_{n_sequences}.txt'
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            seqs = [line.strip() for line in f if line.strip()]
        print(f'  Loaded {len(seqs)} cached ({cache_tag})')
        return seqs

    print(f'  Downloading from UniProt ({cache_tag})...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Python/mine_v2'})
    with urllib.request.urlopen(req) as response:
        raw = response.read().decode('ascii')

    proteins, current = [], []
    for line in raw.split('\n'):
        if line.startswith('>'):
            if current:
                proteins.append(''.join(current))
            current = []
        elif line.strip():
            current.append(line.strip().upper())
    if current:
        proteins.append(''.join(current))

    valid_aa = set(AMINO_ACIDS)
    valid = [p for p in proteins if len(p) >= min_length
             and all(c in valid_aa for c in p[:min_length])]

    rng = np.random.default_rng(seed)
    if len(valid) > n_sequences:
        idx = rng.choice(len(valid), n_sequences, replace=False)
        valid = [valid[i] for i in idx]

    seqs = [p[:min_length] for p in valid[:n_sequences]]
    with open(cache_file, 'w') as f:
        f.write('\n'.join(seqs))
    print(f'  Got {len(seqs)} sequences ({cache_tag})')
    return seqs


def compute_aa_features(sequences):
    '''Compute amino acid composition + biophysical features.'''
    features = []
    for seq in sequences:
        L = len(seq)
        aa_freq = np.array([seq.count(aa) / L for aa in AMINO_ACIDS])
        len_feat = np.array([L / 1000.0])
        charge = (seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')) / L
        hydro_aa = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,
                    'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,
                    'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,
                    'W':-0.9,'Y':-1.3}
        hydro = np.mean([hydro_aa.get(aa, 0) for aa in seq])
        extra = np.array([charge, hydro])
        features.append(np.concatenate([aa_freq, len_feat, extra]))
    return np.array(features, dtype=np.float32)


def pad_protein(signal_seq, target_length, rng):
    pad_needed = target_length - len(signal_seq)
    if pad_needed <= 0:
        return signal_seq[:target_length]
    pad = ''.join(rng.choice(list(AMINO_ACIDS), size=pad_needed))
    return signal_seq + pad


def cleanup_gpu():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# Download proteins
print("Downloading protein sequences...")
SIGNAL_LENGTH = 50
ecoli_seqs = download_proteins(ECOLI_URL, config['n_sequences'], SIGNAL_LENGTH, RANDOM_SEED, 'mine_ecoli')
human_seqs = download_proteins(HUMAN_URL, config['n_sequences'], SIGNAL_LENGTH, RANDOM_SEED, 'mine_human')

all_protein_signals = ecoli_seqs + human_seqs
protein_labels = np.array([0]*len(ecoli_seqs) + [1]*len(human_seqs))

X_bio_protein = compute_aa_features(all_protein_signals)
species_onehot = np.zeros((len(protein_labels), 2), dtype=np.float32)
species_onehot[np.arange(len(protein_labels)), protein_labels] = 1.0
X_bio_protein = np.concatenate([X_bio_protein, species_onehot], axis=1)

print(f"\nProtein ground truth: {X_bio_protein.shape} (AA comp + charge + hydro + species)")
print(f"Total sequences: {len(all_protein_signals)} ({len(ecoli_seqs)} E.coli + {len(human_seqs)} Human)")

## PCA + MI Ceiling

In [ ]:
from sklearn.decomposition import PCA

MINE_PCA_DIM = 50

def pca_reduce_embeddings(X_emb, n_components=MINE_PCA_DIM, seed=RANDOM_SEED):
    '''Reduce embedding dimensionality via PCA for stable MINE estimation.'''
    if X_emb.shape[1] <= n_components:
        print(f"    PCA: skipped (dim={X_emb.shape[1]} <= {n_components})")
        return X_emb, None

    actual_components = min(n_components, X_emb.shape[0], X_emb.shape[1])
    pca = PCA(n_components=actual_components, random_state=seed)
    X_reduced = pca.fit_transform(X_emb).astype(np.float32)
    var_explained = pca.explained_variance_ratio_.sum()
    print(f"    PCA: {X_emb.shape[1]}d -> {actual_components}d "
          f"({var_explained:.1%} variance retained)")
    return X_reduced, pca


def calibrate_mi_ceiling(X_bio, config, device='cuda', label='', noise_scale=0.1):
    '''Estimate MI ceiling by running MINE with X_bio as both X and X_hat.'''
    print(f"\n  Calibrating MI ceiling for {label}...")
    rng = np.random.default_rng(RANDOM_SEED)
    X_perfect = X_bio + noise_scale * rng.standard_normal(X_bio.shape).astype(np.float32)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio, X_perfect, config, device=device,
        label=f'Ceiling-{label}'
    )
    print(f"  MI Ceiling ({label}): {mean_mi:.4f} +/- {std_mi:.4f} nats")
    return mean_mi, std_mi


print(f"PCA reduction: all embeddings reduced to {MINE_PCA_DIM}d before MINE")
print("MI ceiling calibration: run MINE(X_bio, X_bio + noise) for each distribution")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MI CEILING CALIBRATION: Protein')
print('='*70)

mi_ceiling_protein, mi_ceiling_protein_std = calibrate_mi_ceiling(
    X_bio_protein, config, device=device, label='Protein'
)

print(f'\nCeiling: Protein = {mi_ceiling_protein:.4f} +/- {mi_ceiling_protein_std:.4f} nats')

## Random Baseline

In [ ]:
import time

print('\n' + '='*70)
print('RANDOM BASELINE: Protein Distribution')
print('='*70)

rng_rand = np.random.default_rng(RANDOM_SEED)

X_emb_rand_protein = rng_rand.standard_normal(
    (X_bio_protein.shape[0], MINE_PCA_DIM)
).astype(np.float32)

t0 = time.time()
traces, finals, rand_mi_protein, rand_std_protein = run_mine_with_ci(
    X_bio_protein, X_emb_rand_protein, config, device=device,
    label='Random-Protein')

noise_floor_protein = rand_mi_protein + 2 * rand_std_protein
print(f'\nNoise floor (protein): {noise_floor_protein:.4f} nats')

## ProtMamba Embedding Function

In [ ]:
import torch, gc, glob
import torch.nn as nn

def make_protmamba_embed_fn(batch_size=8, signal_length=None):
    import sys; sys.path.append(os.path.abspath('ProtMamba-ssm'))
    from ProtMamba_ssm.modules import MambaLMHeadModelwithPosids, load_model
    from ProtMamba_ssm.utils import AA_TO_ID

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    for pattern in [f'{PROTMAMBA_WEIGHTS_DIR}/**/checkpoint*',
                    f'{PROTMAMBA_WEIGHTS_DIR}/**/*.pt']:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            ckpt = os.path.dirname(matches[0]) if os.path.isfile(matches[0]) else matches[0]
            break
    else:
        ckpt = PROTMAMBA_WEIGHTS_DIR

    model = load_model(ckpt, model_class=MambaLMHeadModelwithPosids,
                       device=device, dtype=torch.float16, checkpoint_mixer=False)
    model.eval()
    n_layers = model.config.n_layer
    aa_to_id = AA_TO_ID.copy()
    unk_id = aa_to_id.get('X', 0)

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i+batch_size]
            max_len = max(len(s) for s in batch)
            ids = [[aa_to_id.get(aa, unk_id) for aa in s] + [0]*(max_len-len(s)) for s in batch]
            tokens = torch.tensor(ids, dtype=torch.long, device=device)
            pos_ids = torch.arange(max_len, device=device).unsqueeze(0).expand(len(batch), -1)
            hs = model(input_ids=tokens, save_layer=[n_layers], position_ids=pos_ids)
            for j, seq in enumerate(batch):
                pool_len = min(signal_length, len(seq)) if signal_length else len(seq)
                layer_out = hs[n_layers][j, :pool_len, :]
                if isinstance(layer_out, np.ndarray):
                    pooled = layer_out.mean(0).astype(np.float32)
                else:
                    pooled = layer_out.mean(0).cpu().float().numpy()
                embeddings.append(pooled)
            if (i // batch_size + 1) % 20 == 0:
                print(f"    Batch {i//batch_size+1}", end='\r')
        return np.array(embeddings)

    @torch.no_grad()
    def compute_perplexity(sequences, max_seqs=100):
        '''Compute autoregressive perplexity as task competence control.

        ProtMamba forward with save_layer=[] returns CausalLMOutput namedtuple:
          out[0] = loss (None)
          out[1] = logits tensor (batch, seq_len, vocab_size)

        Model runs in float16, so we cast logits to float32 before
        computing cross-entropy to avoid overflow/nan.
        '''
        total_loss = 0.0
        total_tokens = 0
        n_eval = min(len(sequences), max_seqs)

        for i in range(n_eval):
            seq = sequences[i]
            ids = [aa_to_id.get(aa, unk_id) for aa in seq]
            tokens = torch.tensor([ids], dtype=torch.long, device=device)
            pos_ids = torch.arange(len(ids), device=device).unsqueeze(0)

            out = model(input_ids=tokens, position_ids=pos_ids, save_layer=[])
            logits = out[1].float()  # cast fp16 -> fp32 to prevent overflow

            shift_logits = logits[:, :-1, :].contiguous()
            shift_targets = tokens[:, 1:].contiguous()

            loss_fn = nn.CrossEntropyLoss(reduction='sum')
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)),
                          shift_targets.view(-1))
            total_loss += loss.item()
            total_tokens += shift_targets.numel()

            if (i + 1) % 20 == 0:
                print(f"    Perplexity: {i+1}/{n_eval}", end='\r')

        if total_tokens == 0:
            print("    WARNING: Could not compute perplexity")
            return None

        avg_loss = total_loss / total_tokens
        ppl = np.exp(avg_loss)
        print(f"\n    Perplexity: {ppl:.2f} (avg CE: {avg_loss:.4f}, {total_tokens} tokens)")
        return ppl

    pool_desc = f'signal_length={signal_length}' if signal_length else 'full sequence'
    print(f"ProtMamba ready ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params, pool: {pool_desc})")
    return embed, compute_perplexity, model

## Run MINE: Regime III (ProtMamba)

In [ ]:
import time
import pandas as pd

mine_results = []
mine_traces = {}

NOTEBOOK_TAG = 'protmamba'

print('\n' + '='*70)
print('REGIME III: ProtMamba (Geometric Vacuity)')
print('='*70)

embed_fn_pm, perplexity_fn_pm, model_pm = make_protmamba_embed_fn(
    batch_size=8, signal_length=SIGNAL_LENGTH
)

# Embeddings first (fp16 is fine for embeddings)
for seq_len in config['seq_lengths']:
    print(f'\n--- ProtMamba L={seq_len} (pooling first {SIGNAL_LENGTH} tokens) ---')
    rng_pad = np.random.default_rng(RANDOM_SEED)
    padded = [pad_protein(s, seq_len, rng_pad) for s in all_protein_signals]

    cache_emb = f'{CACHE_DIR}/protmamba_mine_sigpool_L{seq_len}.npy'
    if os.path.exists(cache_emb):
        X_emb = np.load(cache_emb)
        print(f'  Loaded cached embeddings: {X_emb.shape}')
    else:
        print(f'  Computing embeddings ({len(padded)} seqs)...')
        X_emb = embed_fn_pm(padded)
        X_emb = np.nan_to_num(X_emb, nan=0.0, posinf=1e6, neginf=-1e6)
        np.save(cache_emb, X_emb)
        print(f'  Saved: {cache_emb} | Shape: {X_emb.shape}')

    t0 = time.time()
    X_emb_pca, _ = pca_reduce_embeddings(X_emb)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio_protein, X_emb_pca, config, device=device, label=f'ProtMamba-L{seq_len}')

    mine_results.append({
        'model': 'ProtMamba', 'regime': 'III', 'seq_length': seq_len,
        'mi_mean': mean_mi, 'mi_std': std_mi, 'mi_runs': finals,
        'time_min': (time.time()-t0)/60,
        'ground_truth': 'protein', 'mi_ceiling': mi_ceiling_protein,
        'mi_normalized': mean_mi / max(mi_ceiling_protein, 1e-8),
        'perplexity': None,  # filled in below
    })
    mine_traces[f'ProtMamba-L{seq_len}'] = traces

# Now compute perplexity in fp32 (fp16 overflows in lm_head -> NaN logits)
print('\n--- Task Competence Control: Perplexity (fp32) ---')
model_pm.float()  # cast entire model fp16 -> fp32

import sys; sys.path.append(os.path.abspath('ProtMamba-ssm'))
from ProtMamba_ssm.utils import AA_TO_ID
aa_to_id = AA_TO_ID.copy()
unk_id = aa_to_id.get('X', 0)

total_loss = 0.0
total_tokens = 0
n_eval = min(len(all_protein_signals), 100)

with torch.no_grad():
    for i in range(n_eval):
        seq = all_protein_signals[i]
        ids = [aa_to_id.get(aa, unk_id) for aa in seq]
        tokens = torch.tensor([ids], dtype=torch.long, device='cuda')
        pos_ids = torch.arange(len(ids), device='cuda').unsqueeze(0)

        out = model_pm(input_ids=tokens, position_ids=pos_ids, save_layer=[])
        logits = out[1]

        shift_logits = logits[:, :-1, :].contiguous()
        shift_targets = tokens[:, 1:].contiguous()

        loss = nn.CrossEntropyLoss(reduction='sum')(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_targets.view(-1))

        if not torch.isnan(loss):
            total_loss += loss.item()
            total_tokens += shift_targets.numel()

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{n_eval}', end='\r')

avg_loss = total_loss / total_tokens
ppl_pm = np.exp(avg_loss)
print(f'\n  Perplexity: {ppl_pm:.2f} (avg CE: {avg_loss:.4f}, {total_tokens} tokens)')

# Patch perplexity into results
for row in mine_results:
    if row['model'] == 'ProtMamba':
        row['perplexity'] = float(ppl_pm)

# Random baseline
mine_results.append({
    'model': 'Random_Protein', 'regime': 'baseline', 'seq_length': 0,
    'mi_mean': rand_mi_protein, 'mi_std': rand_std_protein, 'mi_runs': [],
    'time_min': 0,
    'ground_truth': 'protein', 'mi_ceiling': mi_ceiling_protein,
    'mi_normalized': rand_mi_protein / max(mi_ceiling_protein, 1e-8),
})

del model_pm
cleanup_gpu()

## Save Results

In [ ]:
import pandas as pd
import json as _json

# Save results to Drive
results_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_results_{PHASE}.json'
with open(results_file, 'w') as f:
    # Convert numpy types for JSON serialization
    clean_results = []
    for r in mine_results:
        cr = {}
        for k, v in r.items():
            if isinstance(v, np.floating):
                cr[k] = float(v)
            elif isinstance(v, np.integer):
                cr[k] = int(v)
            elif isinstance(v, np.ndarray):
                cr[k] = v.tolist()
            elif isinstance(v, list):
                cr[k] = [float(x) if isinstance(x, np.floating) else x for x in v]
            else:
                cr[k] = v
        clean_results.append(cr)
    _json.dump(clean_results, f, indent=2)

# Save traces
traces_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_traces_{PHASE}.json'
clean_traces = {}
for k, v in mine_traces.items():
    clean_traces[k] = [list(map(float, t)) for t in v]
with open(traces_file, 'w') as f:
    _json.dump(clean_traces, f)

print(f'Saved results: {results_file}')
print(f'Saved traces: {traces_file}')

df = pd.DataFrame(clean_results)
print('\n' + df.to_string(index=False))